# Instalação de Dependências

In [3]:
# ============================================================
# CÉLULA 1 — Instalação de Dependências
# ============================================================
!pip install ultralytics==8.3.* -q
!pip install opencv-python-headless -q

import ultralytics
ultralytics.checks()
print("✅ Dependências instaladas com sucesso!")

Ultralytics 8.3.253 🚀 Python-3.12.13 torch-2.11.0+cpu CPU (Intel Xeon CPU @ 2.20GHz)
Setup complete ✅ (2 CPUs, 12.7 GB RAM, 20.3/107.7 GB disk)
✅ Dependências instaladas com sucesso!


# Configuração da API do Kaggle

In [4]:
# ============================================================
# CÉLULA 2 — Configurar credenciais do Kaggle
# Execute esta célula ANTES de baixar o dataset.
# Você precisa do arquivo kaggle.json da sua conta Kaggle:
# https://www.kaggle.com/settings → API → Create New Token
# ============================================================
from google.colab import files
import os

print("📤 Faça upload do arquivo kaggle.json")
uploaded = files.upload()

os.makedirs("/root/.kaggle", exist_ok=True)
os.rename("kaggle.json", "/root/.kaggle/kaggle.json")
os.chmod("/root/.kaggle/kaggle.json", 0o600)
print("✅ Credenciais do Kaggle configuradas!")

📤 Faça upload do arquivo kaggle.json


Saving kaggle.json to kaggle.json
✅ Credenciais do Kaggle configuradas!


# Download e Extração do Dataset

In [5]:
# ============================================================
# CÉLULA 3 — Download do Dataset de Defeitos de Madeira
# ============================================================
import os
from pathlib import Path

DATASET_DIR = Path("/content/wood_dataset_raw")
DATASET_DIR.mkdir(parents=True, exist_ok=True)

print("⏬ Baixando dataset... (pode demorar alguns minutos)")
os.chdir(DATASET_DIR)
!kaggle datasets download -d nomihsa965/large-scale-image-dataset-of-wood-surface-defects --unzip
os.chdir("/content")
print("✅ Dataset extraído!")

# Listando o conteúdo
for p in sorted(DATASET_DIR.iterdir()):
    print(f"  📁 {p.name}")

⏬ Baixando dataset... (pode demorar alguns minutos)
Dataset URL: https://www.kaggle.com/datasets/nomihsa965/large-scale-image-dataset-of-wood-surface-defects
License(s): Attribution 4.0 International (CC BY 4.0)
100% 3.09G/3.09G [00:42<00:00, 77.5MB/s]

✅ Dataset extraído!
  📁 Bounding Boxes - YOLO Format - 1
  📁 Images - 1


# Exploração e Mapeamento do Dataset

In [6]:
# ============================================================
# CÉLULA 4 — Mapeamento de imagens e labels
# ============================================================
import random
import numpy as np
import cv2
from pathlib import Path
from tqdm import tqdm

SEED = 42
random.seed(SEED)
np.random.seed(SEED)

DATASET_RAW = Path("/content/wood_dataset_raw")
IMG_EXTS = {".jpg", ".jpeg", ".png", ".bmp", ".tif", ".tiff", ".webp"}
MIN_BOX_PX = 4

def find_files(root, exts=None):
    root = Path(root)
    files = []
    for p in root.rglob("*"):
        if p.is_file():
            if exts is None or p.suffix.lower() in exts:
                files.append(p)
    return sorted(files)

def safe_imread(path):
    img = cv2.imread(str(path))
    return img

def read_yolo_txt(path):
    rows = []
    try:
        with open(path) as f:
            for line in f:
                parts = line.strip().split()
                if len(parts) == 5:
                    cls, xc, yc, bw, bh = map(float, parts)
                    rows.append((int(cls), xc, yc, bw, bh))
    except Exception:
        pass
    return rows

def keep_box(bw, bh, img_w, img_h, min_px):
    return (bw * img_w >= min_px) and (bh * img_h >= min_px)

# Mapeando imagens
all_images = find_files(DATASET_RAW, IMG_EXTS)
all_labels = find_files(DATASET_RAW, {".txt"})

image_map = {p.stem: p for p in all_images}
label_map = {p.stem: p for p in all_labels}

print(f"📸 Total de imagens encontradas: {len(image_map)}")
print(f"🏷️  Total de labels encontradas: {len(label_map)}")
print(f"🔗 Imagens com label correspondente: {len(set(image_map) & set(label_map))}")

📸 Total de imagens encontradas: 4000
🏷️  Total de labels encontradas: 4000
🔗 Imagens com label correspondente: 4000


# Preparação do Dataset (Train / Val)

In [7]:
# ============================================================
# CÉLULA 5 — Organização do Dataset em Estrutura YOLO
# ============================================================
import shutil
import pandas as pd
from sklearn.model_selection import train_test_split

WORK = Path("/content/wood_yolo_dataset")
VAL_SIZE = 0.2
IMG_SZ = 640

for split in ["train", "val"]:
    (WORK / "images" / split).mkdir(parents=True, exist_ok=True)
    (WORK / "labels" / split).mkdir(parents=True, exist_ok=True)

# Coletando registros válidos
records = []
for stem, img_path in tqdm(image_map.items(), desc="Processando imagens"):
    lbl_path = label_map.get(stem)
    img = safe_imread(img_path)
    if img is None:
        continue
    h, w = img.shape[:2]

    cleaned_rows = []
    dropped = 0
    if lbl_path and lbl_path.exists():
        for cls, xc, yc, bw, bh in read_yolo_txt(lbl_path):
            if keep_box(bw, bh, w, h, MIN_BOX_PX):
                cleaned_rows.append((cls, xc, yc, bw, bh))
            else:
                dropped += 1

    has_defect = len(cleaned_rows) > 0
    records.append({
        "stem": stem,
        "image_path": str(img_path),
        "label_path": str(lbl_path) if lbl_path else None,
        "has_defect": int(has_defect),
        "num_boxes": len(cleaned_rows),
        "dropped_boxes": dropped,
        "cleaned_rows": cleaned_rows,
    })

df = pd.DataFrame(records)
print(f"\nDistribuição has_defect:\n{df['has_defect'].value_counts()}")
print(f"Boxes descartadas (muito pequenas): {df['dropped_boxes'].sum()}")

# Split treino/validação estratificado
train_df, val_df = train_test_split(
    df, test_size=VAL_SIZE, random_state=SEED, stratify=df["has_defect"]
)
print(f"\n✅ Treino: {len(train_df)} | Validação: {len(val_df)}")

def copy_split(subset_df, split_name):
    for _, row in tqdm(subset_df.iterrows(), total=len(subset_df), desc=f"Copiando {split_name}"):
        stem = row["stem"]
        img_src = Path(row["image_path"])
        dst_img = WORK / "images" / split_name / img_src.name
        shutil.copy2(img_src, dst_img)

        dst_lbl = WORK / "labels" / split_name / f"{stem}.txt"
        with open(dst_lbl, "w") as f:
            for cls, xc, yc, bw, bh in row["cleaned_rows"]:
                f.write(f"{cls} {xc:.6f} {yc:.6f} {bw:.6f} {bh:.6f}\n")

copy_split(train_df, "train")
copy_split(val_df, "val")
print("✅ Dataset organizado!")

Processando imagens: 100%|██████████| 4000/4000 [01:44<00:00, 38.19it/s]



Distribuição has_defect:
has_defect
1    3612
0     388
Name: count, dtype: int64
Boxes descartadas (muito pequenas): 330

✅ Treino: 3200 | Validação: 800


Copiando val: 100%|██████████| 800/800 [00:05<00:00, 141.28it/s]

✅ Dataset organizado!


# Criação do data.yaml

In [8]:
# ============================================================
# CÉLULA 6 — Criando arquivo de configuração data.yaml
# ============================================================
import yaml

data_yaml = {
    "path": str(WORK),
    "train": "images/train",
    "val": "images/val",
    "nc": 1,
    "names": ["wood_defect"],
}

yaml_path = WORK / "data.yaml"
with open(yaml_path, "w") as f:
    yaml.dump(data_yaml, f, default_flow_style=False, allow_unicode=True)

print("✅ data.yaml criado:")
with open(yaml_path) as f:
    print(f.read())

✅ data.yaml criado:
names:
- wood_defect
nc: 1
path: /content/wood_yolo_dataset
train: images/train
val: images/val



# Treinamento do YOLOv8

In [ ]:
# ============================================================
# CÉLULA 7 — Treinamento do modelo YOLOv8
# Modelo: yolov8m.pt (médio) — bom equilíbrio velocidade/precisão
# Para GPU T4 no Colab: ~1-2h para 50 epochs
# ============================================================
from ultralytics import YOLO

MODEL_NAME = "yolov8m.pt"  # Opções: yolov8n.pt (rápido), yolov8s.pt, yolov8m.pt, yolov8l.pt
EPOCHS = 50                 # Aumente para 100 para melhor resultado
BATCH = 16                  # Reduza para 8 se der OOM
IMG_SZ = 640

model = YOLO(MODEL_NAME)

results = model.train(
    data=str(yaml_path),
    epochs=EPOCHS,
    imgsz=IMG_SZ,
    batch=BATCH,
    workers=2,
    seed=SEED,
    patience=15,           # Early stopping
    optimizer="AdamW",
    lr0=1e-3,
    lrf=0.01,
    warmup_epochs=3,
    cos_lr=True,
    # Augmentações
    hsv_h=0.015,
    hsv_s=0.7,
    hsv_v=0.4,
    flipud=0.3,
    fliplr=0.5,
    mosaic=1.0,
    mixup=0.1,
    copy_paste=0.1,
    # Configurações de saída
    project="/content/runs/wood",
    name="yolov8m_wood",
    exist_ok=True,
    verbose=True,
    plots=True,
)

print(f"\n✅ Treinamento concluído!")
print(f"📁 Resultados salvos em: {results.save_dir}")

# Caminho para o melhor modelo
BEST_MODEL_PATH = Path(results.save_dir) / "weights" / "best.pt"
print(f"🏆 Melhor modelo: {BEST_MODEL_PATH}")

New https://pypi.org/project/ultralytics/8.4.66 available 😃 Update with 'pip install -U ultralytics'
Ultralytics 8.3.253 🚀 Python-3.12.13 torch-2.11.0+cpu CPU (Intel Xeon CPU @ 2.20GHz)
engine/trainer: agnostic_nms=False, amp=True, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.1, copy_paste_mode=flip, cos_lr=True, cutmix=0.0, data=/content/wood_yolo_dataset/data.yaml, degrees=0.0, deterministic=True, device=cpu, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=50, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.3, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.001, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.1, mode=train, model=yolov8m.pt, momentum=0.937, mosaic=1.0, multi_scale=False, name=yolov8m_wood, nbs=64, nms=Fal

# Validação e Métricas

In [ ]:
# ============================================================
# CÉLULA 8 — Validação do modelo e visualização de métricas
# ============================================================
from ultralytics import YOLO
from IPython.display import Image, display
import glob

# Carrega o melhor modelo salvo
model_eval = YOLO(str(BEST_MODEL_PATH))

# Rodando validação
val_results = model_eval.val(
    data=str(yaml_path),
    imgsz=IMG_SZ,
    batch=BATCH,
    conf=0.25,
    iou=0.55,
    plots=True,
    verbose=True,
)

print(f"\n📈 Métricas de Validação:")
print(f"  mAP@50:     {val_results.box.map50:.4f}")
print(f"  mAP@50-95:  {val_results.box.map:.4f}")
print(f"  Precision:  {val_results.box.mp:.4f}")
print(f"  Recall:     {val_results.box.mr:.4f}")

# Exibindo curvas de treinamento
run_dir = Path(results.save_dir)
for img_name in ["results.png", "confusion_matrix.png", "PR_curve.png"]:
    img_path = run_dir / img_name
    if img_path.exists():
        print(f"\n📊 {img_name}:")
        display(Image(str(img_path), width=800))

# Teste em Imagens do Dataset

In [ ]:
# ============================================================
# CÉLULA 9 — Teste em imagens de validação
# ============================================================
import matplotlib.pyplot as plt
import matplotlib.patches as patches
import cv2
import random

model_infer = YOLO(str(BEST_MODEL_PATH))

# Pegando amostras aleatórias de validação
val_imgs = list((WORK / "images" / "val").glob("*"))
sample_imgs = random.sample(val_imgs, min(6, len(val_imgs)))

fig, axes = plt.subplots(2, 3, figsize=(18, 10))
axes = axes.flatten()

for ax, img_path in zip(axes, sample_imgs):
    results_pred = model_infer.predict(
        source=str(img_path),
        conf=0.25,
        iou=0.55,
        imgsz=IMG_SZ,
        verbose=False,
    )
    result = results_pred[0]
    annotated = result.plot()
    annotated_rgb = cv2.cvtColor(annotated, cv2.COLOR_BGR2RGB)
    n_boxes = len(result.boxes)
    status = f"⚠️ {n_boxes} defeito(s)" if n_boxes > 0 else "✅ Sem defeitos"
    ax.imshow(annotated_rgb)
    ax.set_title(f"{img_path.name}\n{status}", fontsize=9, fontweight="bold")
    ax.axis("off")

plt.suptitle("🔍 Detecção de Defeitos em Madeira — Amostras de Validação", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.savefig("/content/wood_detection_samples.png", dpi=150, bbox_inches="tight")
plt.show()
print("✅ Amostras salvas em /content/wood_detection_samples.png")

# Detecção em Tempo Real com Webcam


In [ ]:
# ============================================================
# CÉLULA 10 — Detecção em tempo real via Webcam no Colab
#
# Como funciona:
#   - Captura frame da webcam via JavaScript
#   - Roda inferência YOLOv8
#   - Exibe resultado anotado no Colab
#   - Loop controlado por botão
# ============================================================
from IPython.display import display, Javascript, Image as IPImage
from google.colab.output import eval_js
from base64 import b64decode, b64encode
import numpy as np
import cv2
import io
from PIL import Image as PILImage
from ultralytics import YOLO
import time

model_webcam = YOLO(str(BEST_MODEL_PATH))

CONF_THRESH = 0.30
IOU_THRESH  = 0.55
IMG_SZ_CAM  = 640

def js_to_image(js_reply):
    """Converte base64 do JS para numpy array BGR."""
    image_bytes = b64decode(js_reply.split(",")[1])
    jpg_as_np = np.frombuffer(image_bytes, dtype=np.uint8)
    return cv2.imdecode(jpg_as_np, flags=1)

def array_to_data_url(img_bgr):
    """Converte numpy array BGR para data URL PNG."""
    img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)
    pil_img = PILImage.fromarray(img_rgb)
    buf = io.BytesIO()
    pil_img.save(buf, format="JPEG", quality=85)
    encoded = b64encode(buf.getvalue()).decode("utf-8")
    return f"data:image/jpeg;base64,{encoded}"

# JavaScript para acessar a webcam
js_code_start = Javascript("""
    async function startCamera() {
        const video = document.createElement('video');
        video.width = 640;
        video.height = 480;
        video.autoplay = true;
        video.style.display = 'none';
        document.body.appendChild(video);

        const stream = await navigator.mediaDevices.getUserMedia({video: true});
        video.srcObject = stream;

        window._woodDetectVideo = video;
        window._woodDetectStream = stream;
        return 'Camera iniciada!';
    }
    startCamera();
""")

js_capture = """
    async function captureFrame() {
        const video = window._woodDetectVideo;
        const canvas = document.createElement('canvas');
        canvas.width = video.videoWidth;
        canvas.height = video.videoHeight;
        canvas.getContext('2d').drawImage(video, 0, 0);
        return canvas.toDataURL('image/jpeg', 0.85);
    }
    captureFrame();
"""

js_stop = Javascript("""
    if (window._woodDetectStream) {
        window._woodDetectStream.getTracks().forEach(track => track.stop());
        window._woodDetectVideo.remove();
        window._woodDetectStream = null;
        console.log('Camera parada.');
    }
""")

print("🎥 Iniciando câmera...")
display(js_code_start)
time.sleep(2)  # aguarda a câmera iniciar

N_FRAMES = 30   # número de frames para processar (aumente conforme desejar)
frame_times = []

print(f"🔍 Rodando detecção em {N_FRAMES} frames... (pode levar ~{N_FRAMES*1.5:.0f}s)")
print("─" * 60)

for i in range(N_FRAMES):
    try:
        # Captura frame
        js_reply = eval_js(js_capture)
        frame_bgr = js_to_image(js_reply)

        t0 = time.time()
        # Inferência
        pred_results = model_webcam.predict(
            source=frame_bgr,
            conf=CONF_THRESH,
            iou=IOU_THRESH,
            imgsz=IMG_SZ_CAM,
            verbose=False,
            augment=False,
        )
        t1 = time.time()
        frame_times.append(t1 - t0)

        result = pred_results[0]
        n_det = len(result.boxes)
        annotated = result.plot(line_width=2, font_size=12)

        # Adiciona HUD de status
        h_ann, w_ann = annotated.shape[:2]
        status_color = (0, 50, 200) if n_det > 0 else (0, 180, 60)
        status_text  = f"DEFEITO DETECTADO: {n_det}" if n_det > 0 else "SEM DEFEITOS"
        fps_val = 1.0 / (t1 - t0) if (t1 - t0) > 0 else 0

        # Background do HUD
        overlay = annotated.copy()
        cv2.rectangle(overlay, (0, 0), (w_ann, 60), status_color, -1)
        cv2.addWeighted(overlay, 0.6, annotated, 0.4, 0, annotated)

        cv2.putText(annotated, status_text,  (10, 25),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.8, (255, 255, 255), 2, cv2.LINE_AA)
        cv2.putText(annotated, f"Frame {i+1}/{N_FRAMES} | {fps_val:.1f} FPS",
                    (10, 50), cv2.FONT_HERSHEY_SIMPLEX, 0.55, (220, 220, 220), 1, cv2.LINE_AA)

        # Exibe no Colab
        data_url = array_to_data_url(annotated)
        display(Javascript(f"""
            (() => {{
                let existing = document.getElementById('woodDetectOutput');
                if (!existing) {{
                    existing = document.createElement('img');
                    existing.id = 'woodDetectOutput';
                    existing.style.cssText = 'max-width:100%;border:2px solid #333;border-radius:8px;';
                    document.querySelector('.output').appendChild(existing);
                }}
                existing.src = '{data_url}';
            }})();
        """))

        if n_det > 0:
            confs = result.boxes.conf.cpu().numpy()
            print(f"  ⚠️  Frame {i+1:02d}: {n_det} defeito(s) | conf: {confs.mean():.2f} | {fps_val:.1f} FPS")
        else:
            print(f"  ✅  Frame {i+1:02d}: Sem defeitos detectados | {fps_val:.1f} FPS")

    except Exception as e:
        print(f"  ❌ Frame {i+1} — Erro: {e}")
        continue

# Para a câmera
display(js_stop)

avg_fps = 1.0 / np.mean(frame_times) if frame_times else 0
print("─" * 60)
print(f"\n✅ Sessão encerrada!")
print(f"📊 FPS médio de inferência: {avg_fps:.2f}")
print(f"⏱️  Tempo médio por frame: {np.mean(frame_times)*1000:.1f} ms")

# Salvar Modelo no Google Drive (Opcional)

In [ ]:
# ============================================================
# CÉLULA 11 — Salvar modelo no Google Drive
# ============================================================
from google.colab import drive
import shutil
from pathlib import Path

drive.mount("/content/drive")

DRIVE_SAVE_DIR = Path("/content/drive/MyDrive/wood_defect_model")
DRIVE_SAVE_DIR.mkdir(parents=True, exist_ok=True)

best_pt = BEST_MODEL_PATH
dst = DRIVE_SAVE_DIR / "best_wood_yolov8m.pt"
shutil.copy2(best_pt, dst)
print(f"✅ Modelo salvo em: {dst}")

# Também salva o data.yaml para referência futura
shutil.copy2(yaml_path, DRIVE_SAVE_DIR / "data.yaml")
print(f"✅ data.yaml salvo em: {DRIVE_SAVE_DIR / 'data.yaml'}")

# Carregar Modelo Salvo e Usar Webcam (Sessão Futura)

In [ ]:
# ============================================================
# CÉLULA 12 — Carregar modelo do Drive em sessões futuras
# Execute apenas se já tiver o modelo treinado salvo no Drive
# ============================================================
from google.colab import drive
from ultralytics import YOLO
from pathlib import Path

drive.mount("/content/drive")

SAVED_MODEL = Path("/content/drive/MyDrive/wood_defect_model/best_wood_yolov8m.pt")

if SAVED_MODEL.exists():
    model_loaded = YOLO(str(SAVED_MODEL))
    print(f"✅ Modelo carregado de: {SAVED_MODEL}")
    print("ℹ️  Para usar a webcam, redefina BEST_MODEL_PATH e execute a Célula 10.")
    BEST_MODEL_PATH = SAVED_MODEL
else:
    print("❌ Modelo não encontrado. Execute o treinamento primeiro (Células 1–7).")